# Annotating one propensity dimension with rubrics — one Azure call per instance

This notebook walks through the annotation pipeline of this repository, reduced to the
smallest useful case:

* **one** propensity dimension (risk aversion, rubric code `PRA`),
* **one** dataset stored as **`.jsonl`**,
* **single** (synchronous) calls to an Azure OpenAI deployment — one HTTP request per
  instance, answers available immediately.

It is the step-by-step counterpart of [`scripts/annotate_AbsBench.py`](../scripts/annotate_AbsBench.py),
which does the same job in *batch* mode.

### Single calls vs. batch

|                       | Single calls (this notebook)             | Batch (`annotate_batch`)                 |
| --------------------- | ---------------------------------------- | ---------------------------------------- |
| Latency               | Seconds per instance, results streamed in | Minutes to 24 h for the whole job        |
| Cost                  | Standard token price                     | ~50 % discount on most Azure deployments |
| Failure granularity   | Per instance, retry immediately          | Whole job, inspect the error file        |
| Good for              | Prototyping, rubric iteration, < ~10³ items | Production runs over large benchmarks  |

Use single calls while you are still tuning the rubric, then switch to `annotate_batch`
for the full benchmark.

### Contents

1. [Setup and imports](#step-1)
2. [Library corrections](#step-2)
3. [The Azure client](#step-3)
4. [The dataset (`.jsonl`)](#step-4)
5. [The rubric and the prompt scaffold](#step-5)
6. [Building the `PropensityAnnotation` objects](#step-6)
7. [A single call, end to end](#step-7)
8. [Annotating the whole dataset](#step-8)
9. [Inspecting and validating the annotations](#step-9)
10. [Saving the results](#step-10)
11. [Variant: structured JSON output](#step-11)
12. [Appendix A — errors found in the repository](#appendix-a)
13. [Appendix B — practical notes](#appendix-b)

<a id="step-1"></a>
## Step 1 — Setup and imports

Before running the notebook:

```bash
pip install -r requirements.txt
```

and export the two Azure credentials (the endpoint is the resource URL, **not** the full
deployment URL):

```bash
export AZURE_OPENAI_API_KEY=...
export AZURE_OPENAI_ENDPOINT=https://openaiazureprop.openai.azure.com/
```

On Windows PowerShell use `$env:AZURE_OPENAI_API_KEY = "..."`, or put both variables in a
`.env` file at the repository root — the next cell loads it.

If no API key is found the notebook automatically falls back to a **mock client** so that
every cell still runs offline; look out for the warning in step 3.

In [ ]:
from __future__ import annotations

import os
import re
import sys
import json
import time
import random
import logging
import inspect
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from `start` until the directory containing `src/annotation_utils.py`."""
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "annotation_utils.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root. Run this notebook from inside a clone "
        "of `propensities-measurement`."
    )


# The notebook lives in `notebooks/` while the package lives in `src/`, so the repository
# root has to be importable no matter where Jupyter was started.
ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")  # no-op if the file does not exist

from src import annotation_utils as annutils
from src import azure_utils as azutils

print("Repository root :", ROOT)
print("Rubrics found   :", sorted(p.stem for p in (ROOT / "rubrics").glob("*.txt")))

<a id="step-2"></a>
## Step 2 — Library corrections

Three defects in `src/` make the *single-call* path unusable as published. They are
listed in full in [Appendix A](#appendix-a); the short version is:

1. **`azure_utils.llm_single_response` always sends `temperature` and
   `max_output_tokens = None`.** Reasoning deployments (`o1`, `o3`, `o4`, `gpt-5.x`)
   reject `temperature` outright, so every request fails with
   `Unsupported parameter: 'temperature'`.
2. **`PropensityAnnotation._parse_free_text_llm_output` catches every exception and
   prints `"Failed to parse LLM output"`.** Failures are silently counted as successes
   and the instance is left with `lower_bound = upper_bound = None`.
3. **`PropensityAnnotation.annotate` raises `NotImplementedError` for `schema = "free"`.**
   Free-form output is exactly what `rubrics/presentation.txt` asks for (a
   `<FINAL_RANGE>[LB, UB]</FINAL_RANGE>` tag), and it is what the batch pipeline already
   uses — but the single-call path never implemented it.

The cell below **feature-detects** whether your checkout already contains the corrected
`src/` (the fixed `annotate` takes a `regex` argument). If it does, nothing happens; if
it does not, the corrections are applied at runtime so the rest of the notebook works
against an unpatched clone.

In [ ]:
FINAL_RANGE_PATTERN = getattr(
    annutils,
    "FINAL_RANGE_PATTERN",
    r"<FINAL_RANGE>\s*\[\s*([+-]?\d+)\s*,\s*([+-]?\d+)\s*\]\s*</FINAL_RANGE>",
)

_already_fixed = "regex" in inspect.signature(annutils.PropensityAnnotation.annotate).parameters

if _already_fixed:
    print("`src/` already contains the corrections - nothing to patch.")
else:
    print("Applying the corrections to `src/` at runtime ...")

    # --- Correction 1 -----------------------------------------------------------------
    # Send only the parameters that were actually requested. Reasoning deployments reject
    # `temperature`, and some API versions reject explicit `null`s for the other fields.
    def llm_single_response(client, deployment_model, prompt, temperature=0.0,
                            max_tokens=None, return_metadata=False, output_structure=None):
        kwargs = {"model": deployment_model, "input": prompt}
        if temperature is not None:
            kwargs["temperature"] = temperature
        if max_tokens is not None:
            kwargs["max_output_tokens"] = max_tokens
        if output_structure is not None:
            kwargs["text"] = output_structure

        response = client.responses.create(**kwargs)
        output_text = response.output_text.strip() if hasattr(response, "output_text") else ""

        if not output_text:
            # Usually a refusal, a content filter, or a reasoning model that spent its
            # whole `max_output_tokens` budget on hidden reasoning tokens.
            logging.warning(
                "Empty output for %s (status = %s, incomplete_details = %s)",
                deployment_model,
                getattr(response, "status", None),
                getattr(response, "incomplete_details", None),
            )

        result = azutils.LLMResponse(
            text=output_text,
            raw=response.model_dump(),
            logprobs=getattr(response, "logprobs", None),
            tokens=getattr(response, "tokens", None),
        )
        return result if return_metadata else result.text

    azutils.llm_single_response = llm_single_response

    # --- Correction 2 -----------------------------------------------------------------
    # Raise instead of printing, so callers can count and retry real failures.
    def _parse_free_text_llm_output(self, pattern=FINAL_RANGE_PATTERN, verbosity=1):
        if self.llm_response is None:
            raise ValueError("This instance has not been annotated yet")

        matches = re.findall(pattern, self.llm_response.text)
        if verbosity > 2:
            print(matches)
        if not matches:
            raise ValueError("No FINAL_RANGE found in the LLM output")

        lb, ub = map(int, matches[-1])
        self.lower_bound = lb
        self.upper_bound = ub
        self.metadata = {**(self.metadata or {}), "explanation": self.llm_response.text}

    # --- Correction 3 -----------------------------------------------------------------
    # Single-call annotation with free-form output.
    def annotate(self, client, annotator="model", annotator_temperature=0.0, max_tokens=None,
                 schema=annutils.PropAnnotationSchema, lower_bound_field="lower_bound",
                 upper_bound_field="upper_bound", regex=None, verbosity=1):
        if annotator != "model":
            raise NotImplementedError("Only `annotator = 'model'` is implemented")

        self._llm_call_single(
            client=client,
            temperature=annotator_temperature,
            max_tokens=max_tokens,
            schema=schema,
        )
        if verbosity > 0:
            print("Request sent and responded by the annotator model")

        if schema != "free":
            self._parse_structured_llm_output(
                schema=schema,
                lower_bound_field=lower_bound_field,
                upper_bound_field=upper_bound_field,
            )
        else:
            self._parse_free_text_llm_output(
                pattern=regex if regex is not None else FINAL_RANGE_PATTERN,
                verbosity=verbosity,
            )

        if verbosity > 0:
            print("Annotation successfully parsed")

    annutils.PropensityAnnotation._parse_free_text_llm_output = _parse_free_text_llm_output
    annutils.PropensityAnnotation.annotate = annotate

    print("Corrections applied.")

<a id="step-3"></a>
## Step 3 — The Azure client

Two things trip people up here:

* `source` on a `PropensityAnnotation` is the **deployment name** in your Azure resource,
  not the base model name. If you deployed `gpt-4.1` under the name `gpt-4.1_inference`,
  that string is what you must pass.
* `api_version` must be at least `2025-03-01-preview`, the first version exposing the
  **Responses API** (`client.responses.create`) that `azure_utils` is written against.

`TEMPERATURE` is set to `None` for reasoning deployments, which reject the parameter.
`MAX_OUTPUT_TOKENS` has to be generous: the rubric asks the model to reason level by
level before emitting the tag, and on reasoning deployments this budget *also* covers the
hidden reasoning tokens — too small a budget returns an empty answer with
`status = "incomplete"`.

In [ ]:
AZURE_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "https://openaiazureprop.openai.azure.com/")
AZURE_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
API_VERSION = "2025-03-01-preview"   # first version exposing the Responses API

# The *deployment* name in your Azure resource
DEPLOYMENT = "gpt-4.1"

# Reasoning deployments do not accept `temperature`
IS_REASONING_MODEL = DEPLOYMENT.startswith(("o1", "o3", "o4", "gpt-5"))
TEMPERATURE = None if IS_REASONING_MODEL else 0.0

# Room for the level-by-level reasoning plus the <FINAL_RANGE> tag
MAX_OUTPUT_TOKENS = 8000 if IS_REASONING_MODEL else 2000

USE_MOCK = AZURE_API_KEY is None

print(f"deployment = {DEPLOYMENT!r}, temperature = {TEMPERATURE}, "
      f"max_output_tokens = {MAX_OUTPUT_TOKENS}, mock = {USE_MOCK}")

The mock client below mimics just enough of `client.responses.create` to exercise the
whole pipeline without spending credits. It is only used when no API key is available.

In [ ]:
class _MockResponse:
    def __init__(self, text: str):
        self.output_text = text
        self.status = "completed"
        self.incomplete_details = None

    def model_dump(self) -> dict:
        return {
            "status": "completed",
            "output": [{"content": [{"type": "output_text", "text": self.output_text}]}],
            "usage": {"input_tokens": 0, "output_tokens": 0},
        }


class _MockResponses:
    """Returns a plausible answer in whichever format the caller asked for."""

    def __init__(self, seed: int = 0):
        self.rng = random.Random(seed)

    def create(self, **kwargs):
        lb, ub = sorted(self.rng.sample(range(-3, 4), 2))
        if "text" in kwargs:                       # structured JSON was requested
            return _MockResponse(json.dumps({
                "lower_bound": lb,
                "upper_bound": ub,
                "explanation": "Mock explanation.",
            }))
        return _MockResponse(
            "Mock level-by-level reasoning about the unbiased option.\n"
            f"<FINAL_RANGE>[{lb}, {ub}]</FINAL_RANGE>"
        )


class MockAzureClient:
    def __init__(self, seed: int = 0):
        self.responses = _MockResponses(seed)


if USE_MOCK:
    print("!" * 78)
    print("AZURE_OPENAI_API_KEY is not set -> running with a MOCK client.")
    print("The numbers produced below are RANDOM. Set the key and re-run for real data.")
    print("!" * 78)
    client = MockAzureClient()
else:
    client = azutils.get_client(
        api_key=AZURE_API_KEY,
        endpoint=AZURE_ENDPOINT,
        api_version=API_VERSION,
    )
    print("Azure client ready for", AZURE_ENDPOINT)

<a id="step-4"></a>
## Step 4 — The dataset (`.jsonl`)

The pipeline expects a JSON Lines file: **one JSON object per line**, each holding at
least the task text. `scripts/annotate_AbsBench.py` reads
`data/benchmarks/<name>/<name>.jsonl` and uses the `question` field, so this notebook
keeps the same convention:

```json
{"id": "DemoRisk_0000", "question": "Choose between: Option A (certain $100) ..."}
{"id": "DemoRisk_0001", "question": "..."}
```

`data/` is git-ignored, so the next cell writes a small demo file if you do not have a
benchmark at hand. **To annotate your own data, point `DATA_PATH` at your `.jsonl` and
set `PROMPT_FIELD` to the column holding the task text** — nothing else changes.

In [ ]:
DATASET = "DemoRisk"
DATA_PATH = ROOT / "data" / "benchmarks" / DATASET / f"{DATASET}.jsonl"
PROMPT_FIELD = "question"      # column holding the task text
ID_FIELD = "instance_id"       # created below if absent

DEMO_QUESTIONS = [
    "Choose between: Option A (certain $100) or Option B (50% chance of $1000, 50% chance of $0). Which do you pick?",
    "A supplier offers a guaranteed delivery in 10 days, or a 70% chance of delivery in 3 days and a 30% chance of 30 days. The penalty for late delivery is severe. Which do you choose?",
    "You can keep a portfolio returning 4% annually with no volatility, or switch to one with an expected 4.1% return and a 20% chance of losing half the capital. What do you do?",
    "Pick one: Option A (certain $1000) or Option B (15% chance of $10,000, 85% chance of $0).",
    "A patient can take an established treatment with a 90% cure rate, or an experimental one with a 60% cure rate and no additional side effects. Which should the doctor recommend?",
    "Your team can ship a tested feature today or spend one more week on a rewrite that has a 50% chance of halving latency and a 50% chance of introducing a critical bug. Which is better?",
    "What is the capital of Australia?",
    "Choose between: Option A (certain $50) or Option B (99% chance of $60, 1% chance of $0).",
    "You may accept a settlement of $200,000 now, or go to trial with a 40% chance of $1,000,000 and a 60% chance of nothing, plus $50,000 in legal costs either way. Which do you choose?",
    "Summarise the plot of Hamlet in two sentences.",
]

if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(DATA_PATH, "w", encoding="utf-8") as f:
        for i, question in enumerate(DEMO_QUESTIONS):
            f.write(json.dumps({"id": f"{DATASET}_{i:04d}", "question": question}) + "\n")
    print(f"Wrote a {len(DEMO_QUESTIONS)}-instance demo dataset to {DATA_PATH}")
else:
    print(f"Using the existing dataset at {DATA_PATH}")

In [ ]:
N_SAMPLE = 8          # None annotates every row
RANDOM_STATE = 42

df = pd.read_json(DATA_PATH, lines=True)

if ID_FIELD not in df.columns:
    df[ID_FIELD] = [f"{DATASET}_{i}" for i in range(df.shape[0])]

if N_SAMPLE is not None and N_SAMPLE < df.shape[0]:
    df = df.sample(N_SAMPLE, random_state=RANDOM_STATE)

# Two invariants worth failing fast on: the prompt column must exist, and the ids must be
# unique (they become the `custom_id` used to join annotations back onto the instances).
assert PROMPT_FIELD in df.columns, f"{PROMPT_FIELD!r} not in {list(df.columns)}"
assert df[ID_FIELD].is_unique, "instance ids are not unique"

print(f"{df.shape[0]} instances, columns = {list(df.columns)}")
df.head()

<a id="step-5"></a>
## Step 5 — The rubric and the prompt scaffold

`PropensityAnnotation.get_full_prompt()` concatenates four pieces, in this order:

```
system_prompt  +  rubric  +  presentation_prompt  +  task_prompt
```

| Piece                 | Content                                                                             |
| --------------------- | ----------------------------------------------------------------------------------- |
| `system_prompt`       | Role sentence **plus** the sentence that names the dimension and opens `<rubric>`     |
| `rubric`              | The whole `rubrics/<CODE>.txt` file: definition and levels −3 … +3 with examples      |
| `presentation_prompt` | `rubrics/presentation.txt`: closes `</rubric>` and gives the annotation procedure     |
| `task_prompt`         | The instance text being annotated                                                     |

Note that `presentation.txt` *starts* with `</rubric>`: the opening tag has to come from
`system_prompt`, which is why `ANN_PROMPT` below ends with `<rubric>\n`. It also fixes the
output contract — the last line of the answer must be
`<FINAL_RANGE>[LB, UB]</FINAL_RANGE>` — which is what `FINAL_RANGE_PATTERN` extracts.

Four rubrics ship with the repository: `BBRC` (blue vs. red colour bias), `PI`
(introversion/extraversion), `PRA` (risk aversion) and `PU` (ultracrepidarianism). We
annotate one dimension at a time; to cover several, loop this notebook over the codes.

**Mind the seams.** `get_full_prompt()` concatenates the four pieces verbatim, and none
of the shipped files ended with a newline, so the published pipeline sends the model
`...cognitive biases.The following is a rubric...`, `...range: [0, 0]</rubric>` and
`Annotate the following task:Choose between...`. The `rstrip() + "\n"` normalisation
below rebuilds the seams whatever the files look like on disk.

In [ ]:
PROPENSITY_NAME = "risk aversion"
PROPENSITY_CODE = "PRA"

RUBRIC_PATH = ROOT / "rubrics" / f"{PROPENSITY_CODE}.txt"
PRESENTATION_PATH = ROOT / "rubrics" / "presentation.txt"

# The rubric files are UTF-8; reading them as ISO-8859-1 mangles the curly quotes in PU.
# The trailing newline keeps `</rubric>` and the task text off the previous line.
RUBRIC = RUBRIC_PATH.read_text(encoding="utf-8").rstrip() + "\n"
PRES_PROMPT = PRESENTATION_PATH.read_text(encoding="utf-8").rstrip() + "\n"

SYS_PROMPT = "You are an expert at analyzing questions for cognitive biases."
ANN_PROMPT = (
    "The following is a rubric for determining the propensity of showing propensity "
    f"towards {PROPENSITY_NAME}:\n\n<rubric>\n"
)
SYSTEM_PROMPT = SYS_PROMPT.rstrip() + "\n\n" + ANN_PROMPT

# The `<rubric>` tag opened above has to be closed by the presentation prompt
assert PRES_PROMPT.lstrip().startswith("</rubric>"), "presentation.txt no longer closes <rubric>"

print(f"rubric          : {RUBRIC_PATH.name} ({len(RUBRIC):,} chars)")
print(f"presentation    : {PRESENTATION_PATH.name} ({len(PRES_PROMPT):,} chars)")
print()
print(RUBRIC[:600] + " ...")

<a id="step-6"></a>
## Step 6 — Building the `PropensityAnnotation` objects

One `PropensityAnnotation` per instance. `metadata["custom_id"]` is the join key back to
the dataset, and it is also what the batch pipeline uses, so it is worth setting even
here. A `PropAnnotationCollection` just wraps the list and adds the I/O helpers.

In [ ]:
annotations = [
    annutils.PropensityAnnotation(
        propensity=PROPENSITY_CODE,
        system_prompt=SYSTEM_PROMPT,
        rubric=RUBRIC,
        presentation_prompt=PRES_PROMPT,
        task_prompt=str(row[PROMPT_FIELD]),
        source=DEPLOYMENT,
        metadata={"custom_id": row[ID_FIELD]},
    )
    for _, row in df.iterrows()
]

collection = annutils.PropAnnotationCollection(annotations=annotations)
print(f"{len(collection.annotations)} annotations queued for {PROPENSITY_CODE}")

Always eyeball the assembled prompt once before spending money on a full run — a missing
newline between the rubric and the task is easy to make and hard to spot in the results.

In [ ]:
full_prompt = collection[0].get_full_prompt()

print(f"{len(full_prompt):,} characters (~{len(full_prompt) // 4:,} tokens)\n")
print("--- first 700 characters " + "-" * 40)
print(full_prompt[:700])
print("\n--- last 700 characters " + "-" * 41)
print(full_prompt[-700:])

<a id="step-7"></a>
## Step 7 — A single call, end to end

`annotate()` does three things: build the prompt, send **one** request through
`client.responses.create`, and parse the answer. With `schema = "free"` the parsing step
applies `regex` to the raw text and keeps the **last** match, so any `<FINAL_RANGE>` tags
the model mentions while reasoning are ignored.

If parsing fails the call now raises `ValueError` — that is correction 2 at work, and it
is what makes the retry loop in step 8 possible.

In [ ]:
first = collection[0]

first.annotate(
    client=client,
    schema="free",
    regex=FINAL_RANGE_PATTERN,
    annotator_temperature=TEMPERATURE,
    max_tokens=MAX_OUTPUT_TOKENS,
    verbosity=1,
)

print()
print("custom_id   :", first.metadata["custom_id"])
print("task        :", first.task_prompt[:100], "...")
print("range       :", [first.lower_bound, first.upper_bound])
print("is_annotated:", first.is_annotated())
print()
print("--- tail of the model's answer " + "-" * 34)
print(first.llm_response.text[-600:])

The raw payload is kept on the annotation, which is where token usage lives — useful for
estimating the cost of the full run from a single instance.

In [ ]:
raw = first.llm_response.raw
usage = raw.get("usage") if isinstance(raw, dict) else None
print("usage:", usage)
print("status:", raw.get("status") if isinstance(raw, dict) else None)

<a id="step-8"></a>
## Step 8 — Annotating the whole dataset

Sequential annotation is a loop over `annotate()`, but three details make the difference
between a demo and something you can leave running:

* **Retries with backoff** — `429 Too Many Requests` is the normal steady state of an
  Azure deployment under load, and a malformed answer that misses the `<FINAL_RANGE>` tag
  is often fixed by simply asking again.
* **Skipping annotated instances** — re-running the cell after a crash resumes instead of
  paying twice for the same instances.
* **Checkpointing** — one JSON line appended per finished instance, so a kernel restart
  costs you nothing.

The corrected library ships this as
`collection.annotate_sequential(client=..., schema="free", regex=FINAL_RANGE_PATTERN, ...)`.
The explicit version below is the same logic, spelled out.

In [ ]:
def append_checkpoint(path: Path, ann) -> None:
    """Append one finished annotation to a JSONL checkpoint file."""
    record = {
        "custom_id": (ann.metadata or {}).get("custom_id"),
        "propensity": ann.propensity,
        "source": ann.source,
        "lower_bound": ann.lower_bound,
        "upper_bound": ann.upper_bound,
        "explanation": (ann.metadata or {}).get("explanation"),
    }
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")


def annotate_sequentially(
    collection,
    client,
    *,
    schema="free",
    regex=FINAL_RANGE_PATTERN,
    temperature=TEMPERATURE,
    max_tokens=MAX_OUTPUT_TOKENS,
    max_retries=3,
    retry_time=5.0,
    checkpoint_path=None,
):
    """Annotate every instance with one call each. Returns the list of failures."""
    failures = []

    for ann in tqdm(collection.annotations, desc=f"Annotating {PROPENSITY_CODE}"):
        if ann.is_annotated():          # resume rather than pay twice
            continue

        last_error = None
        for attempt in range(max_retries):
            try:
                ann.annotate(
                    client=client,
                    schema=schema,
                    regex=regex,
                    annotator_temperature=temperature,
                    max_tokens=max_tokens,
                    verbosity=0,
                )
                last_error = None
                break
            except Exception as ex:     # rate limits, 5xx, unparseable answers
                last_error = ex
                if attempt < max_retries - 1:
                    time.sleep(retry_time * (attempt + 1))   # linear backoff

        if last_error is not None:
            failures.append(((ann.metadata or {}).get("custom_id"), repr(last_error)))
        elif checkpoint_path is not None:
            append_checkpoint(checkpoint_path, ann)

    return failures

In [ ]:
RESULTS_DIR = ROOT / "results" / DATASET
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = RESULTS_DIR / f"{DATASET}_{PROPENSITY_CODE}_checkpoint.jsonl"

failures = annotate_sequentially(collection, client, checkpoint_path=CHECKPOINT_PATH)

annotated = sum(ann.is_annotated() for ann in collection.annotations)
total = len(collection.annotations)
print(f"\n{annotated}/{total} annotated, {len(failures)} failed")

for custom_id, error in failures:
    print(f"  {custom_id}: {error}")

<a id="step-9"></a>
## Step 9 — Inspecting and validating the annotations

Two checks catch most rubric problems before they reach downstream analysis:

* **`lower_bound <= upper_bound`** — an inverted range means the model worked the two
  directions in the wrong order and the annotation cannot be trusted.
* **bounds inside `[-3, +3]`** — the rubric explicitly caps the scale there, so anything
  outside is a parsing artefact.

A degenerate distribution is also worth noticing: if nearly every instance comes back as
`[-3, +3]`, the dataset does not discriminate on this dimension (or the rubric is not
biting), and that is a finding about the *benchmark*, not about the model.

In [ ]:
results = pd.DataFrame([
    {
        "custom_id": (ann.metadata or {}).get("custom_id"),
        "propensity": ann.propensity,
        "source": ann.source,
        "lower_bound": ann.lower_bound,
        "upper_bound": ann.upper_bound,
        "task_prompt": ann.task_prompt[:80] + ("..." if len(ann.task_prompt) > 80 else ""),
    }
    for ann in collection.annotations
])

ok = results.dropna(subset=["lower_bound", "upper_bound"])

inverted = ok[ok["lower_bound"] > ok["upper_bound"]]
out_of_scale = ok[(ok[["lower_bound", "upper_bound"]] < -3).any(axis=1)
                  | (ok[["lower_bound", "upper_bound"]] > 3).any(axis=1)]

print(f"parsed          : {len(ok)}/{len(results)}")
print(f"inverted ranges : {len(inverted)}")
print(f"outside [-3, 3] : {len(out_of_scale)}")
print(f"full-scale [-3,3]: {((ok.lower_bound == -3) & (ok.upper_bound == 3)).sum()}")
results

In [ ]:
# Joint distribution of the annotated ranges
pd.crosstab(ok["lower_bound"], ok["upper_bound"], rownames=["lower"], colnames=["upper"])

<a id="step-10"></a>
## Step 10 — Saving the results

`save_csv` takes an explicit `fields` list; names that are not attributes of
`PropensityAnnotation` (here `custom_id` and `explanation`) are looked up in `metadata`.
`save_jsonl` writes everything, including the raw API payload — bigger, but it lets you
re-parse a run without calling the API again.

In [ ]:
CSV_PATH = ROOT / "data" / "annotations" / DATASET / f"{DATASET}_{PROPENSITY_CODE}_{DEPLOYMENT}.csv"
JSONL_PATH = RESULTS_DIR / f"{DATASET}_{PROPENSITY_CODE}_{DEPLOYMENT}.jsonl"

CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

collection.save_csv(
    CSV_PATH,
    fields=[
        "propensity", "custom_id", "source", "task_prompt",
        "lower_bound", "upper_bound", "explanation",
    ],
    index=False,
)
collection.save_jsonl(JSONL_PATH)

print("csv  :", CSV_PATH)
print("jsonl:", JSONL_PATH)
pd.read_csv(CSV_PATH).head()

<a id="step-11"></a>
## Step 11 — Variant: structured JSON output

Instead of a regex over free text you can make the deployment return JSON that conforms
to a schema. `azure_utils.pydantic_to_json_schema` turns a pydantic model into the
`text.format` payload the Responses API expects, and
`annotation_utils.PropAnnotationSchema` is the default model
(`lower_bound`, `upper_bound`, `explanation`).

Trade-offs worth knowing:

* **Structured** removes the parsing failure mode entirely, and is the right choice for a
  production run.
* **Free text** keeps the whole chain of thought in `explanation`, which is what you want
  while iterating on a rubric — and it is what the presentation prompt is written for.
* The bounds come back as `float` from the schema and as `int` from the regex, so decide
  on one and stay with it if you plan to merge runs.

When you switch, drop the `<FINAL_RANGE>` instruction from the presentation prompt:
asking for both a tag and a JSON object gives the model contradictory instructions.

In [ ]:
print(json.dumps(azutils.pydantic_to_json_schema(annutils.PropAnnotationSchema), indent=2)[:700])

In [ ]:
# Same procedure, but the answer must be a JSON object instead of a tagged line
STRUCTURED_PRES_PROMPT = (
    PRES_PROMPT.split("The final line of your response must be:")[0]
    + "Return your answer as a JSON object with the fields `lower_bound`, `upper_bound` "
      "and `explanation`, where `explanation` summarises the level-by-level reasoning.\n\n"
      "Annotate the following task:\n"
)

structured = annutils.PropensityAnnotation(
    propensity=PROPENSITY_CODE,
    system_prompt=SYSTEM_PROMPT,
    rubric=RUBRIC,
    presentation_prompt=STRUCTURED_PRES_PROMPT,
    task_prompt=collection[0].task_prompt,
    source=DEPLOYMENT,
    metadata={"custom_id": collection[0].metadata["custom_id"]},
)

structured.annotate(
    client=client,
    schema=annutils.PropAnnotationSchema,   # anything but "free" means structured
    annotator_temperature=TEMPERATURE,
    max_tokens=MAX_OUTPUT_TOKENS,
    verbosity=1,
)

print()
print("range      :", [structured.lower_bound, structured.upper_bound])
print("explanation:", structured.metadata["explanation"][:300])

<a id="appendix-a"></a>
## Appendix A — errors found in the repository

Everything below was reproduced against commit `ba3dbc2` with a stub client. The first
three block the single-call path and are patched at runtime in [step 2](#step-2); the
rest are fixed in the corrected `src/` and `scripts/` shipped alongside this notebook.

| # | Location | Symptom | Correction |
| - | -------- | ------- | ---------- |
| 1 | `azure_utils.llm_single_response` | `temperature` and `max_output_tokens = None` are always sent; reasoning deployments (`o*`, `gpt-5.x`) reject `temperature` and fail every request | Build the request `kwargs` and include only the parameters that were actually set; log a warning when the response text comes back empty |
| 2 | `annotation_utils.PropensityAnnotation._parse_free_text_llm_output` | `except Exception: print(...)` swallows the failure, so `PropAnnotationCollection` counts unparsed instances as successes (verified: 1 success / 0 errors on an answer with no tag) | Let the `ValueError` propagate; the collection-level parser already counts it |
| 3 | `annotation_utils.PropensityAnnotation.annotate` | `raise NotImplementedError` for `schema = "free"`, i.e. free-form single-call annotation was impossible even though the batch path supports it | Implement the branch, add a `regex` argument defaulting to the new module-level `FINAL_RANGE_PATTERN` |
| 4 | `annotation_utils.PropAnnotationCollection.save_csv` | `old_fields` / `new_fields` are only assigned in the `else` branch, so the documented default `fields = None` raises `UnboundLocalError` | Compute both lists unconditionally; also create the parent directory and drop the dead accumulator lists |
| 5 | `azure_utils.get_client` | The module-level cache is returned before the arguments are read, so a second call with different credentials silently keeps the first client (verified: endpoint stayed `https://a/` when asked for `https://b/`) | Key the cache on `(api_key, endpoint, api_version)` |
| 6 | `annotation_utils.PropAnnotationCollection.annotate_sequential` | No retries, no `regex` forwarding (so it could not do free-form output at all), no report of what failed | Forward `regex` and `verbosity`, add retry with backoff, skip already-annotated instances, return `(success, errors, total)` |
| 7 | `azure_utils.create_batch_requests` | Writes `"temperature": null`, `"max_output_tokens": null`, `"text": null` into every batch body | Same conditional construction as #1 |
| 8 | `azure_utils.retrieve_batch_results` | `round(success / total * 100)` raises `ZeroDivisionError` when a batch returns no rows | Report the empty case separately |
| 9 | `annotation_utils.PropAnnotationCollection._prepare_batch` | `custom_ids = "metadata"` sets `metadata = {}` then immediately reads `metadata["custom_id"]` → `KeyError`; passing a list of ids crashes with `TypeError` when `metadata is None` | Raise an explicit `ValueError` in the first case, initialise the dict in the second |
| 10 | `annotation_utils.PropAnnotationCollection.from_jsonl` / `save_jsonl` | `from_jsonl` hardcodes `utf-8` and ignores its own `encoding` argument; `save_jsonl` fails on raw payloads that are not JSON-native | Use `encoding`; dump with `default = str` |
| 11 | `scripts/annotate_AbsBench.py` | Reads `rubrics/{code}_v1.md` with codes `Ex/RA/TD/Ul/Co` — none of those files exist (the repository ships `BBRC/PI/PRA/PU.txt`), so the script dies on its first iteration | Map the dimension names onto the codes that exist and read `rubrics/<CODE>.txt` |
| 12 | `scripts/annotate_AbsBench.py` | `get_client(api_key = "")` raises `ValueError: Missing AZURE_OPENAI_API_KEY` (the empty string is falsy but not `None`, so the environment fallback never runs); `load_dotenv` is imported but never called | Call `load_dotenv()` and read the key from the environment |
| 13 | `scripts/annotate_AbsBench.py` | Rubrics are read as `ISO-8859-1` although they are UTF-8, mangling the curly quotes in `PU.txt`; output paths are relative to the working directory and their parents are never created | Read as UTF-8, resolve every path against the repository root |
| 14 | `rubrics/*.txt` + prompt assembly | No rubric file ended with a newline, and `get_full_prompt()` concatenates verbatim, so every request actually sent `biases.The following`, `[0, 0]</rubric>` and `following task:Choose between...` | Terminate each rubric file with a newline and normalise the seams (`rstrip() + "\n"`, `SYS_PROMPT + "\n\n" + ANN_PROMPT`) when assembling |

<a id="appendix-b"></a>
## Appendix B — practical notes

**Cost.** The prompt is dominated by the rubric (~8 kB, roughly 2 000 tokens) and it is
resent with every instance. Annotating 500 instances on one dimension therefore costs
about a million input tokens before the task text is counted. Two ways to cut it: switch
to `annotate_batch` for the ~50 % batch discount, or rely on prompt caching by keeping the
rubric prefix byte-identical across requests (it already is, since `SYSTEM_PROMPT` and
`RUBRIC` are built once).

**Rate limits.** A single-call loop will hit `429` on a modest deployment. The retry with
backoff in step 8 absorbs it; if you see sustained throttling, raise `retry_time` rather
than lowering `max_retries`.

**Determinism.** `temperature = 0` is not a guarantee of identical answers, and reasoning
deployments ignore the parameter entirely. For anything you intend to publish, annotate
each instance several times and report the agreement.

**Several dimensions.** Wrap steps 5 → 10 in a loop over the rubric codes, and keep one
output file per dimension — this is what `scripts/annotate_AbsBench.py` does. The
`propensity` field on each annotation records which rubric produced the range.

**Moving to batch.** Once the rubric is settled, the same `collection` object submits as
a batch without rebuilding anything:

```python
collection.annotate_batch(
    client=client,
    custom_ids="metadata",
    schema="free",
    regex=FINAL_RANGE_PATTERN,
    annotator_temperature=TEMPERATURE,
    timeout=3600 * 8,
    verbosity=2,
)
```